# Fine-tune MiniLM for IT Ticket Routing

Fine-tunes `sentence-transformers/all-MiniLM-L6-v2` as a 7-class sequence classifier on the Kaggle IT ticket dataset.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
!pip install -q -U "transformers>=4.41.0" accelerate evaluate
!pip install -q datasets scikit-learn huggingface_hub joblib

import os
os.kill(os.getpid(), 9)  # restart runtime to activate updated packages

In [ ]:
import transformers
print(transformers.__version__)  # should be >= 4.41.0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
os.makedirs('/content/data', exist_ok=True)
shutil.copy('/content/drive/MyDrive/tickets.csv', '/content/data/tickets.csv')
print('tickets.csv loaded')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

CATEGORY_MAP = {
    'Hardware':               'Infrastructure',
    'Administrative rights':  'Access Management',
    'Access':                 'Access Management',
    'Storage':                'Storage',
    'HR Support':             'HR Support',
    'Purchase':               'Procurement',
    'Internal Project':       'Internal Project',
    'Miscellaneous':          'General IT',
}
CATEGORIES = sorted(set(CATEGORY_MAP.values()))
LABEL2ID   = {c: i for i, c in enumerate(CATEGORIES)}
ID2LABEL   = {i: c for c, i in LABEL2ID.items()}

df = pd.read_csv('/content/data/tickets.csv')
df.columns = [c.strip() for c in df.columns]
text_col  = next(c for c in df.columns if 'document' in c.lower() or 'text' in c.lower())
label_col = next(c for c in df.columns if 'topic' in c.lower() or 'label' in c.lower())
df = df[[text_col, label_col]].rename(columns={text_col: 'text', label_col: 'raw_label'})
df['text']     = df['text'].astype(str).str.strip()
df['label']    = df['raw_label'].map(CATEGORY_MAP)
df = df.dropna(subset=['label', 'text'])
df = df[df['text'].str.len() > 5]
df['label_id'] = df['label'].map(LABEL2ID)

print(f'{len(df):,} tickets — {df["label"].nunique()} classes')
print(df['label'].value_counts().to_string())

In [ ]:
train_df, tmp     = train_test_split(df, test_size=0.20, stratify=df['label_id'], random_state=42)
val_df,   test_df = train_test_split(tmp, test_size=0.50, stratify=tmp['label_id'], random_state=42)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'train {len(train_df):,}  val {len(val_df):,}  test {len(test_df):,}')

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

BASE_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer  = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenise_split(df):
    ds = Dataset.from_dict({'text': df['text'].tolist(), 'labels': df['label_id'].tolist()})
    return ds.map(
        lambda b: tokenizer(b['text'], truncation=True, padding='max_length', max_length=128),
        batched=True, batch_size=512,
    )

train_ds = tokenise_split(train_df)
val_ds   = tokenise_split(val_df)
test_ds  = tokenise_split(test_df)

In [ ]:
import torch
from torch.nn import CrossEntropyLoss
from sklearn.utils.class_weight import compute_class_weight
import evaluate as hf_evaluate
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

# class weights to balance the imbalanced training set
raw_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(CATEGORIES)),
    y=train_df['label_id'].values,
)
class_weights = torch.tensor(raw_weights, dtype=torch.float32)
print('class weights:', {c: round(w, 3) for c, w in zip(CATEGORIES, raw_weights)})

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        logits  = outputs.logits
        loss_fct = CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

metric = hf_evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels, average='macro')

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(CATEGORIES),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

STEPS_PER_EPOCH = len(train_ds) // 64
WARMUP_STEPS    = int(0.1 * STEPS_PER_EPOCH * 5)

args = TrainingArguments(
    output_dir='/content/finetuned_minilm',
    num_train_epochs=5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    learning_rate=2e-5,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=100,
    fp16=True,
    report_to='none',
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()
print(f'best val macro-F1: {trainer.state.best_metric:.4f}')

In [ ]:
from sklearn.metrics import classification_report

preds_out = trainer.predict(test_ds)
preds     = np.argmax(preds_out.predictions, axis=-1)
labels    = preds_out.label_ids

print(classification_report(labels, preds, target_names=CATEGORIES))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import os

os.makedirs('/content/plots', exist_ok=True)

cm = confusion_matrix(labels, preds, normalize='true')
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
fig.colorbar(im, ax=ax)

tick_marks = np.arange(len(CATEGORIES))
ax.set_xticks(tick_marks); ax.set_yticks(tick_marks)
ax.set_xticklabels(CATEGORIES, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(CATEGORIES, fontsize=9)

thresh = cm.max() / 2
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, f'{cm[i, j]:.2f}', ha='center', va='center', fontsize=8,
                color='white' if cm[i, j] > thresh else 'black')

ax.set_ylabel('True label', fontsize=11)
ax.set_xlabel('Predicted label', fontsize=11)
ax.set_title('Normalised Confusion Matrix', fontsize=12)
plt.tight_layout()
plt.savefig('/content/plots/confusion_matrix.png', dpi=130, bbox_inches='tight')
plt.show()

from google.colab import files
files.download('/content/plots/confusion_matrix.png')

In [ ]:
from huggingface_hub import login
login(token='hf_YOUR_TOKEN_HERE')

HF_REPO = 'starlord0104/ticket-routing-minilm-finetuned'
trainer.model.push_to_hub(HF_REPO, private=False)
tokenizer.push_to_hub(HF_REPO, private=False)
print(f'https://huggingface.co/{HF_REPO}')

In [ ]:
from transformers import pipeline

clf = pipeline('text-classification', model=HF_REPO, top_k=1)

tests = [
    ('Procurement',      'Please raise a purchase order for five monitors and keyboards for new starters.'),
    ('Infrastructure',   'My laptop screen is black and will not turn on.'),
    ('HR Support',       'New hire starting Monday — set up email and system access.'),
    ('Access Management','I am locked out of my admin account, need a password reset.'),
    ('Storage',          'Drive is full, I cannot save any files.'),
    ('General IT',       'The printer on the third floor has stopped working.'),
    ('Internal Project', 'Please reassign the Q3 project tasks to the new team lead.'),
]

print(f'{"Expected":22s}  {"Predicted":22s}  Conf')
print('-' * 60)
for expected, text in tests:
    result = clf(text)[0][0]
    mark = '✓' if result['label'] == expected else '✗'
    print(f'{mark}  {expected:22s}  {result["label"]:22s}  {result["score"]:.2%}')